In [0]:
from pyspark.sql import functions as F

def common_silver_cleaner(df):
    # 1. Trim all string columns
    for column_name, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(column_name, F.trim(F.col(column_name)))

    # 2. Convert common invalid values to NULL
    invalid_values = ["", "NA", "N/A", "NULL", "null"]

    for column_name, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(column_name,
                    F.when(
                        F.upper(F.col(column_name)).isin([v.upper() for v in invalid_values]),
                        F.lit(None)
                    ).otherwise(F.col(column_name))
                )

    # 3. Add processing timestamp
    df = df.withColumn("_silver_processed_timestamp", F.current_timestamp())

    return df